## Imports

In [ ]:
import numpy as np
import xarray as xr
import copy
import src.utils
import src.lme_utils
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cmocean
import os
import pathlib
import cartopy.crs as ccrs
import pandas as pd
import scipy.stats
import tqdm
import time
import xesmf as xe

## Set seaborn plotting style
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})
sns.set_palette("colorblind")

## initialize RNG
rng = np.random.default_rng(seed=100)

## Funcs

In [ ]:
## specify lons/lats for E/W boxes
LONS_W = [120, 200]
LONS_E = [200, 280]


## funcs to plot boxes
def plot_wbox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_W, lats=[-5, 5], **kwargs)


def plot_ebox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_E, lats=[-5, 5], **kwargs)


## funcs to average over regions
def get_eq_avg(data, lons):
    """average over equatorial region and longitudes"""
    return data.sel(longitude=slice(*lons), latitude=slice(-5, 5)).mean(
        ["longitude", "latitude"]
    )


def get_e(data):
    return get_eq_avg(data, LONS_E)


def get_w(data):
    return get_eq_avg(data, LONS_W)


def get_dx(data):
    return get_e(data) - get_w(data)


def add_gridlines(axs):
    """func to add gridlines to axs object"""

    for ax in axs[:-1, 0]:
        ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[],
            ylocs=[-30, 0, 30],
        )
    for ax in axs[-1]:
        gl_ = ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[50, 120 - 160],
            # xlocs=[],
            ylocs=[-30, 0, 30],
        )
        gl_.top_labels = False
        # gl_.bottom_labels = True
    # gl_.left_labels = False

    return


def plot_level(ax, data, level, ls="-", c="white"):
    """plot single level on hovmoller"""
    ax.contour(
        data.longitude,
        data.latitude,
        data,
        levels=[level],
        colors=c,
        linestyles=ls,
        zorder=10,
        transform=ccrs.PlateCarree(),
    )
    return


import cartopy.crs as ccrs


def make_cb_range(dx, nlevs):
    amp = dx * nlevs - dx / 2

    lev1 = np.arange(0, amp + dx / 2, dx) + dx / 2
    lev0 = -lev1[::-1]
    return np.concatenate([lev0, lev1])


def plot_pr_diff(ax, data, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.balance_r",
        levels=make_cb_range(dx, nlev),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def plot_sst_sigma(ax, data, lev_min=0.3, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.amp",
        levels=np.arange(lev_min, lev_min + dx * (nlev + 1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def plot_pr(ax, data, lev_min=1, dx=1, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.rain",
        levels=np.arange(lev_min, lev_min + dx * (nlev + 1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp

## Data loading

In [ ]:
def sel_area(data, lon_range=None, lat_range=None):
    """select area on t grids"""

    ## find in lon range
    if lon_range is None:
        lon_idx = np.array([True for _ in data.nlon])
    else:
        lon_idx = (
            ((data.TLONG >= lon_range[0]) & (data.TLONG <= lon_range[1]))
            .any("nlat")
            .values
        )

    if lat_range is None:
        lat_idx = np.array([True for _ in data.nlat])
    else:
        lat_idx = (
            ((data.TLAT >= lat_range[0]) & (data.TLAT <= lat_range[1]))
            .any("nlon")
            .values
        )

    return data.isel(nlon=lon_idx, nlat=lat_idx)


def area_avg(data, varname, lon_range=None, lat_range=None):
    """average over area"""

    ## trim data
    data_ = sel_area(data, lon_range=lon_range, lat_range=lat_range)

    ## get dims to sum over
    integrate = lambda x: (x * data_["dA"]).sum(["nlon", "nlat"])

    return integrate(data_[varname]) / integrate(1.0)


def get_eli_helper(exceeds_thresh):
    """compute ELI from mask of threshold exceedance"""

    ## sum and count longitudes exceeding thresh
    lon = exceeds_thresh.longitude
    longitude_sum = (exceeds_thresh * lon).sum(["longitude", "latitude"])
    longitude_count = exceeds_thresh.sum(["longitude", "latitude"])

    ## eli is average longitude
    eli = longitude_sum / longitude_count

    return eli


def get_eli(rsst, rsst_thresh=0, max_lat=15):
    """compute ELI from tropical SST data"""

    ## get equatorial Pac. SST
    rsst_pac = rsst.sel(longitude=slice(120, 280), latitude=slice(-max_lat, max_lat))

    ## get boolean array where SST exceeds thresh
    exceeds_thresh0 = rsst_pac >= rsst_thresh

    ## compute initial ELI estimate
    eli0 = get_eli_helper(exceeds_thresh0)

    return eli0

## Load the data

### init. cluster

In [ ]:
from dask.distributed import LocalCluster, Client

cluster = LocalCluster(n_workers=2)
client = Client(cluster)
client

### Load pre-computed EOFs

In [ ]:
## truncate to NEOFS
NEOFS = 30

## save filepath for eofs
SAVE_FP = pathlib.Path("/glade/work/kcarr/lme_data")
eofs_save_fp = pathlib.Path(SAVE_FP, "snr", "eofs_sst_trop_deseason.nc")

## specify dims
T_DIM = "time"
DIMS = [T_DIM, "member"]


if eofs_save_fp.is_file():
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    eofs_ = xr.open_dataset(eofs_save_fp, decode_times=time_coder)

else:
    print("file doesn't exist")


## truncate
eofs_ = eofs_.isel(mode=slice(None, NEOFS))

In [ ]:
## compute inner products
n_testmode = 10
idx = dict(mode=slice(None, n_testmode))
QtQ = xr.dot(
    eofs_["Q"].isel(idx),
    eofs_["Q"].isel(idx).rename({"mode": "mode_out"}),
    dims=DIMS,
)

ZtZ = xr.dot(
    eofs_["Z"].isel(idx).fillna(0),
    eofs_["Z"].isel(idx).fillna(0).rename({"mode": "mode_out"}),
    dims=["lat", "lon"],
)

## check they're identities
eye = np.eye(n_testmode)
print(np.allclose(ZtZ.values, eye))
print(np.allclose(QtQ.values, eye))

In [ ]:
from scipy.signal import butter, lfilter, freqz, filtfilt


def butter_lowpass(cutoff, fs, order=5):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="low", analog=False)
    return b, a


def butter_lowpass_filter(data, cutoff, fs, order=5, axis=-1):
    b, a = butter_lowpass(cutoff, fs, order=order)
    y = filtfilt(b, a, data, axis=axis)
    return y.squeeze()

In [ ]:
## do low-pass filtering
Qtilde = copy.deepcopy((eofs_["Q"]).transpose(..., DIMS[0]))

fs = 12 if "time" in DIMS else 1

Qtilde.values = butter_lowpass_filter(
    Qtilde.values[None, :], cutoff=1 / 10, fs=fs, axis=-1
)
eofs_["Qtilde"] = Qtilde

In [ ]:
## plot results
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(eofs_["Q"].isel(mode=0, member=0).isel({T_DIM: slice(None, 500 * 12)}))
ax.plot(eofs_["Qtilde"].isel(mode=0, member=0).isel({T_DIM: slice(None, 500 * 12)}))

plt.show()

In [ ]:
## covariance
N = int(len(eofs_.stack(s=DIMS).s))
QtQ = 1 / N * (eofs_["Qtilde"] * eofs_["Qtilde"].rename({"mode": "mode_"})).sum(DIMS)

## eigendecomp
Lambda_sqr, U = np.linalg.eigh(QtQ.values)
U = U[:, ::-1]
Lambda_sqr = Lambda_sqr[::-1]

## put in xarray dataset
eofs_tilde = xr.Dataset(
    data_vars=dict(
        U=(("mode", "mode_tilde"), U),
        Lambda=("mode_tilde", np.sqrt(Lambda_sqr)),
    ),
    coords=dict(mode=eofs_.mode.values, mode_tilde=eofs_.mode.values),
)

## get timeseries timeseries
eofs_tilde["tk"] = xr.dot(eofs_tilde["U"], eofs_["Q"], dim="mode")
eofs_tilde["tk_tilde"] = xr.dot(eofs_tilde["U"], eofs_["Qtilde"], dim="mode")

# get patterns
eofs_tilde["P"] = xr.dot(
    eofs_["Z"].fillna(0),
    eofs_["sigma"],
    eofs_tilde["U"],
    dim="mode",
)

## check tk's are orthgonal
TtT = xr.dot(
    eofs_tilde["tk"],
    eofs_tilde["tk"].rename({"mode_tilde": "mode"}),
    dim=DIMS,
)

## do tests
# print(np.allclose(eofs_tilde["P"], eofs_tilde["P_test"]))
print(np.allclose(TtT.values, np.eye(TtT.shape[0])))

#### Plot structure of EOFs

In [ ]:
fig = plt.figure(figsize=(7, 7), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=25, lon_range=(40, 350))
axs = src.utils.subplots_with_proj(fig, nrows=3, ncols=1, format_func=format_func)


for j in range(3):
    cp00 = plot_pr_diff(
        axs[j, 0],
        eofs_tilde["P"]
        .isel(mode_tilde=j)
        .rename({"lat": "latitude", "lon": "longitude"}),
        dx=3e1,
    )

    ## label
    axs[j, 0].set_title(f"Mode {j}", loc="left")

## add labels
# add_gridlines(axs)
bbox = dict(boxstyle="round", facecolor="white", alpha=1)
legend_kwargs = dict(x=0.01, y=0.02, fontdict=dict(size=12, color="gray"), bbox=bbox)
for ax in axs.flatten():
    ax.set_aspect("auto")
    src.utils.plot_box(ax, lons=[65, 150], lats=[-25, 25], c="k", ls="--")

## colorbars
cb_kwargs = dict(label=r"$^{\circ}$C", pad=0.03)
# fig.colorbar(cp00, ax=axs[0,0], ticks=[-3.6,0,3.6], **cb_kwargs)
# fig.colorbar(cp10, ax=axs[1,0], ticks=[-1.8,0,1.8], **cb_kwargs)
add_gridlines(axs)
plt.show()

#### Timeseries

In [ ]:
plt.plot(eofs_tilde["Lambda"])

In [ ]:
sel = lambda x: x.isel(member=0, mode_tilde=0).isel({T_DIM: slice(None, None)})
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(-sel(eofs_tilde["tk"]))
ax.plot(-sel(eofs_tilde["tk_tilde"]))
plt.show()

#### Compare to AAM

In [ ]:
## annual average
if "time" in DIMS:
    tk_ann = eofs_tilde["tk_tilde"].resample({"time": "YS"}).mean()
else:
    tk_ann = eofs_tilde["tk_tilde"]

## scatter
fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(
    aam.sel(y0=slice(900, 1849)).isel(y0=slice(10, None)).transpose("member", "y0"),
    tk_ann.isel(mode_tilde=0).isel({T_DIM: slice(10, None)}).transpose("member", T_DIM),
    s=10,
)

plt.show()

## timeseries
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(aam.sel(y0=slice(900, 1849)).isel(member=0), label="AAM")
ax.plot(tk_ann.isel(mode_tilde=0, member=0) * 2e2, label="Custom")
ax.set_ylim([-2.2, 2.2])
ax.set_xlim([10, 510])
plt.show()

#### Plot cov. w/SST

In [ ]:
def prep_spatial(x, t_dim):
    """prep spatial data for regression"""

    if t_dim == "time":
        t_idx = dict(time=slice("0900-01", "1849-12"))
        x_total = x["forced"].sel(t_idx) + x["anom"].sel(t_idx)
        x_total = x_total - x_total.mean(["time", "member"])

        ## remove seasonal cycle
        x_grouped = x_total.groupby("time.month")
        x_total = x_grouped - x_grouped.mean(["time", "member"])
        x_total = x_total.assign_coords({"time": eofs_tilde.time})

        ## annual mean
        x_total_ann = x_total.groupby("time.year").mean()

    else:
        x_total_ann = prep(x["anom"]).mean("season")
        # x_total_ann = stitch_monsoon(x["anom"])

    return x_total_ann.fillna(0)

In [ ]:
## annual mean
sst_total_ann = prep_spatial(sst, t_dim=T_DIM)
pr_total_ann = prep_spatial(pr, t_dim=T_DIM)

if T_DIM == "time":
    tk_ann = eofs_tilde["tk"].groupby("time.year").mean()
else:
    tk_ann = eofs_tilde["tk"]

sum_dims = ["year", "member"] if "time" in DIMS else ["y0", "member"]

## compute SST pattern)
P_sst = xr.dot(sst_total_ann, tk_ann.isel(mode_tilde=slice(None, 3)), dim=sum_dims)
P_pr = xr.dot(pr_total_ann, tk_ann.isel(mode_tilde=slice(None, 3)), dim=sum_dims)

In [ ]:
## specify plot data and interval
p, DX = P_sst, 6e0
# p, DX = -P_pr, 3e1


fig = plt.figure(figsize=(7, 7), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=25, lon_range=(40, 280))
axs = src.utils.subplots_with_proj(fig, nrows=3, ncols=1, format_func=format_func)


for j in range(3):
    cp = plot_sst(axs[j, 0], p.isel(mode_tilde=j), dx=DX)
    axs[j, 0].set_title(f"Mode {j}", loc="left")

## add labels
# add_gridlines(axs)
bbox = dict(boxstyle="round", facecolor="white", alpha=1)
legend_kwargs = dict(x=0.01, y=0.02, fontdict=dict(size=12, color="gray"), bbox=bbox)
for ax in axs.flatten():
    ax.set_aspect("auto")
    src.utils.plot_box(ax, lons=[120, 170], lats=[-5, 5], c="w", ls="--")
    src.utils.plot_box(ax, lons=[40, 280], lats=[-1e-3, 1e-3], c="k", ls="--", lw=0.8)

## colorbars
cb_kwargs = dict(label=r"$^{\circ}$C", pad=0.03)
plt.show()

Scratch

In [ ]:
## get principal components (normalized so that q.T@q=1)
Q = scores_norm / np.sqrt(N)

## check normalization
print(np.allclose(np.ones(10), (Q**2).sum(["member", "y0"]).values))

In [ ]:
import scipy.signal

# def make_lanczos_filter(cutoff, fs, order=5):
#     nyq = 0.5 * fs
#     normal_cutoff = cutoff / nyq
#     b, a = butter(order, normal_cutoff, btype="low", analog=False)
#     return b, a


# def apply_lanczos_filter(data, cutoff, fs=1):
#     b, a = butter_lowpass(cutoff, fs, order=order)
#     y = filtfilt(b, a, data, axis=1)
#     return y

In [ ]:
help(scipy.signal.windows.lanczos)

In [ ]:
help(scipy.signal.filtfilt)

In [ ]:
help(scipy.signal.sosfiltfilt)